In [1]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_groq import ChatGroq

from langchain_cohere import CohereRerank
from dotenv import load_dotenv

load_dotenv()

C:\Users\Vidwa\AppData\Local\Temp\ipykernel_19780\4005067682.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever
c:\Users\Vidwa\OneDrive\Desktop\23BCE7320\RAG Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
%pip install langchain_cohere

Note: you may need to restart the kernel to use updated packages.


In [3]:
chunks = [
    # Tesla - Financial & Production
    "Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.",
    "Tesla's automotive gross margin improved to 19.3% this quarter.",
    "Tesla Cybertruck production ramp begins in 2024 with initial deliveries.",
    "Tesla announced plans to expand Gigafactory production capacity.",
    "Tesla stock price reached new highs following earnings announcement.",
    "Tesla's energy storage business grew 40% year-over-year.",
    "Tesla continues to lead in electric vehicle market share globally.",
    "Tesla Model Y became the best-selling vehicle worldwide.",
    "Tesla reported strong free cash flow generation of $7.5 billion.",
    "Tesla's Full Self-Driving revenue increased significantly.",
    
    # Microsoft - Development & Acquisitions
    "Microsoft acquired GitHub for $7.5 billion in 2018.",
    "Microsoft's cloud revenue Azure grew 29% year-over-year.",
    "Microsoft announced new AI features for Visual Studio Code.",
    "Microsoft Teams integration with GitHub enhances developer workflow.",
    "Microsoft's developer tools division sees strong adoption.",
    "Microsoft acquired Activision Blizzard for $68.7 billion.",
    "Microsoft's productivity suite gained 50 million new users.",
    "Microsoft announced new Surface devices for developers.",
    "Microsoft's AI Copilot features expand to more development tools.",
    "Microsoft's enterprise solutions drive revenue growth.",
    
    # NVIDIA - AI & Hardware
    "NVIDIA's data center revenue reached $47.5 billion annually.",
    "NVIDIA's H100 GPUs see unprecedented demand for AI training.",
    "NVIDIA announced next-generation Blackwell architecture.",
    "NVIDIA's gaming revenue declined due to crypto market changes.",
    "NVIDIA's automotive AI platform partnerships expanded.",
    "NVIDIA's AI chip shortage affects cloud providers.",
    "NVIDIA stock valuation exceeds $2 trillion market cap.",
    "NVIDIA's CUDA platform dominates AI development.",
    "NVIDIA announced new AI inference chips for edge computing.",
    "NVIDIA's partnership with major cloud providers strengthens.",
    
    # Google/Alphabet - AI & Cloud
    "Google's AI investments total over $100 billion in recent years.",
    "Google Cloud revenue grew 35% reaching $8.4 billion quarterly.",
    "Google announced Gemini AI model competing with GPT-4.",
    "Google's search advertising revenue remains strong at $59 billion.",
    "Google's Workspace products integrate advanced AI features.",
    "Google announced quantum computing breakthroughs.",
    "Google's autonomous vehicle division Waymo expands operations.",
    "Google's AI research published breakthrough papers.",
    "Google's cloud AI services see enterprise adoption.",
    "Google faces regulatory scrutiny over AI dominance.",
    
    # Noisy/Less Relevant Chunks
    "The Tesla coil was invented by Nikola Tesla in 1891.",
    "Microsoft Excel spreadsheet formulas can be complex for beginners.",
    "NVIDIA Shield TV streaming device gets software update.",
    "Google Maps navigation improved with real-time traffic data.",
    "Production delays affected multiple manufacturing sectors.",
    "Financial markets showed volatility during earnings season.",
    "Revenue recognition standards changed for software companies.",
    "Hardware components face supply chain constraints globally.",
    "Development tools market grows with remote work trends.",
    "AI research requires significant computational resources.",
    "Quarterly reports show mixed results across tech sector.",
    "Stock market analysts upgrade technology sector ratings.",
    "Cloud computing adoption accelerates in enterprise market.",
    "Data center construction increases globally.",
    "Semiconductor shortage impacts various industries.",
    "Electric vehicle charging infrastructure expands rapidly.",
    "Software development productivity tools gain popularity.",
    "Machine learning frameworks become more accessible.",
    "Enterprise software licensing models evolve.",
    "Technology conferences showcase latest innovations."
]

print(f"Created {len(chunks)} sample chunks for demonstration")

Created 60 sample chunks for demonstration


In [4]:
documents = [Document(page_content=chunk, metadata={"source": f"chunk_{i}"}) for i, chunk in enumerate(chunks)]

In [5]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(
    documents = documents,
    embedding = embedding_model,
    collection_metadata={"hnsw:space":"cosine"}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1339.27it/s]


In [6]:
print("Vector Search")
print()
vector_retriever = vector_store.as_retriever(search_kwargs={"k":15})
test_query = "space exploration company"

test_docs = vector_retriever.invoke(test_query)
for i,docs in enumerate(test_docs,1):
    print(f"Document{i}: {docs.page_content}")

Vector Search

Document1: Microsoft acquired GitHub for $7.5 billion in 2018.
Document2: Microsoft acquired Activision Blizzard for $68.7 billion.
Document3: Google announced quantum computing breakthroughs.
Document4: Google announced Gemini AI model competing with GPT-4.
Document5: Google's autonomous vehicle division Waymo expands operations.
Document6: Google's AI investments total over $100 billion in recent years.
Document7: Tesla's energy storage business grew 40% year-over-year.
Document8: Technology conferences showcase latest innovations.
Document9: Development tools market grows with remote work trends.
Document10: Google's AI research published breakthrough papers.
Document11: Google's search advertising revenue remains strong at $59 billion.
Document12: NVIDIA's automotive AI platform partnerships expanded.
Document13: NVIDIA's data center revenue reached $47.5 billion annually.
Document14: Tesla announced plans to expand Gigafactory production capacity.
Document15: Google

In [7]:
print("BM25 Retriever")

bm25_retriever = BM25Retriever.from_documents(documents=documents)
bm25_retriever.k = 15

BM25 Retriever


In [8]:
test_query = "Tesla"
test_docs = bm25_retriever.invoke(test_query)
for i,doc in enumerate(test_docs,1):
    print(f"Document{i}: {doc}")

Document1: page_content='The Tesla coil was invented by Nikola Tesla in 1891.' metadata={'source': 'chunk_40'}
Document2: page_content='Tesla Model Y became the best-selling vehicle worldwide.' metadata={'source': 'chunk_7'}
Document3: page_content='Tesla announced plans to expand Gigafactory production capacity.' metadata={'source': 'chunk_3'}
Document4: page_content='Tesla stock price reached new highs following earnings announcement.' metadata={'source': 'chunk_4'}
Document5: page_content='Tesla reported strong free cash flow generation of $7.5 billion.' metadata={'source': 'chunk_8'}
Document6: page_content='Tesla Cybertruck production ramp begins in 2024 with initial deliveries.' metadata={'source': 'chunk_2'}
Document7: page_content='Tesla continues to lead in electric vehicle market share globally.' metadata={'source': 'chunk_6'}
Document8: page_content='Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.' metadata={'source': 'chunk_0'}
Document9: page_content='

In [13]:
print("EnsembleRetriever")

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever,bm25_retriever],
    weights=[0.7,0.3]
)

test_query = "Tesla financial performance and production updates"
hybrid_docs = hybrid_retriever.invoke(test_query)

for i,doc in enumerate(hybrid_docs,1):
    print(f"Document{i}: {doc.page_content}")

EnsembleRetriever
Document1: Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.
Document2: Tesla announced plans to expand Gigafactory production capacity.
Document3: Tesla reported strong free cash flow generation of $7.5 billion.
Document4: Tesla stock price reached new highs following earnings announcement.
Document5: Tesla continues to lead in electric vehicle market share globally.
Document6: Tesla Cybertruck production ramp begins in 2024 with initial deliveries.
Document7: Tesla Model Y became the best-selling vehicle worldwide.
Document8: The Tesla coil was invented by Nikola Tesla in 1891.
Document9: Financial markets showed volatility during earnings season.
Document10: Tesla's automotive gross margin improved to 19.3% this quarter.
Document11: Tesla's energy storage business grew 40% year-over-year.
Document12: Tesla's Full Self-Driving revenue increased significantly.
Document13: Stock market analysts upgrade technology sector ratings.
Document14: Producti

In [14]:
print("STEP 2: After Cohere Reranking (Top 10)")
print("-"*50)

# Initialize Cohere reranker
reranker = CohereRerank(model="rerank-english-v3.0", top_n=10)

# Rerank the retrieved documents
reranked_docs = reranker.compress_documents(hybrid_docs, query)

# Show reranked results
for i, doc in enumerate(reranked_docs, 1):
    print(f"{i:2d}. {doc.page_content}")

print("\n" + "="*80)
print("ANALYSIS:")
print("✅ Hybrid Search: Mixed relevant and irrelevant results")
print("✅ Reranking: Most relevant Tesla financial/production info at top")
print("✅ Notice how reranking moved the most contextually relevant chunks higher")

# Optional: Show the difference more clearly
print("\n" + "="*80)
print("KEY IMPROVEMENTS AFTER RERANKING:")
print("-"*40)


hybrid_top_5 = [doc.page_content for doc in hybrid_docs[:5]]
reranked_top_5= [doc.page_content for doc in reranked_docs[:5]]

print("BEFORE (Hybrid Top 3):")
for i, content in enumerate(hybrid_top_5, 1):
    print(f"  {i}. {content}")

print("\nAFTER (Reranked Top 3):")
for i, content in enumerate(reranked_top_5, 1):
    print(f"  {i}. {content}")


STEP 2: After Cohere Reranking (Top 10)
--------------------------------------------------
 1. Tesla reported strong free cash flow generation of $7.5 billion.
 2. Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.
 3. Tesla's automotive gross margin improved to 19.3% this quarter.
 4. Tesla announced plans to expand Gigafactory production capacity.
 5. Tesla's energy storage business grew 40% year-over-year.
 6. Tesla continues to lead in electric vehicle market share globally.
 7. Tesla's Full Self-Driving revenue increased significantly.
 8. Tesla Cybertruck production ramp begins in 2024 with initial deliveries.
 9. Tesla stock price reached new highs following earnings announcement.
10. Financial markets showed volatility during earnings season.

ANALYSIS:
✅ Hybrid Search: Mixed relevant and irrelevant results
✅ Reranking: Most relevant Tesla financial/production info at top
✅ Notice how reranking moved the most contextually relevant chunks higher

KEY IMPROVEMEN

In [16]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
import os
print("\n" + "="*80)
print("FINAL: RAG with Reranked Context")
print("-"*40)

# Use top 5 reranked documents for final answer
top_reranked = reranked_docs[:5]

combined_input = f"""Based on the following documents, please answer this question: {test_query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in top_reranked])}

Please provide a clear, helpful answer using only the information from these documents."""


model = ChatGroq(model="openai/gpt-oss-120b",api_key=os.getenv("GROQ_API_KEY"))
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=combined_input),
]

result = model.invoke(messages)
print("Generated Response:")
print(result.content)


FINAL: RAG with Reranked Context
----------------------------------------
Generated Response:
**Tesla – Financial Performance & Production Updates (based on the provided documents)**  

| Metric / Update | Detail (from the documents) |
|-----------------|-----------------------------|
| **Free cash flow** | Generated **$7.5 billion** in free cash flow. |
| **Quarterly revenue** | Achieved a **record $25.2 billion** in revenue for **Q3 2024**. |
| **Automotive gross margin** | Improved to **19.3 %** for the quarter. |
| **Gigafactory production** | **Announced plans to expand Gigafactory production capacity** (no specific numbers given). |
| **Energy storage business** | Grew **40 % year‑over‑year**. |

**Key take‑aways**

- Tesla’s cash generation is strong, with $7.5 billion of free cash flow.
- Revenue reached an all‑time high of $25.2 billion in Q3 2024, indicating robust sales growth.
- The automotive segment’s profitability improved, with the gross margin rising to 19.3 %.
- The 